# IPSA CSBS-BS Scoring Assignments

## tl;dr

The primary deliverable is one Jessi-facing CSV with exactly four columns: `ID`, `Visit Month`, `Examiner`, and `Assigned Scoring Clinician`. It is written only after the live REDCap export contains the configured examiner field and all quality checks pass.

The live full-data export supplies `demo_id`, `csbsbs_examiner`, and `csbsbs_doe`. IDs ending in `--1` or `--2` are automatically generated double-entry validation copies, so the pipeline ignores them entirely for assignments and contact history. The resulting 486-row base-ID population is verified exactly against REDCap report 4692. The run fails closed on a missing field, an unrecognized nonblank examiner value, an invalid evaluation date, or any quality-check failure. No generated or placeholder records are used.


## Context & Methods

For a completed base-ID CSBS visit with a known examiner, the scorer is chosen by Never Seen, Least Visits, Furthest in Time, lower workload, then a seeded random tie-break. Every Emma, Tessa, or Axie name present in a single- or co-scored examiner value is excluded from that visit's scorer candidates. Legitimate examiners outside this three-person scoring pool are preserved and leave all three scorers eligible. Contact counts use all completed CSBS events for base IDs only, deduplicated by child, scoring clinician, and Date of Evaluation; time gaps use actual evaluation dates in days. Double-entry copies never affect counts, time gaps, workload, or assignments.

When the examiner field is present but a particular completed visit is genuinely blank, the scorer is chosen from Emma, Tessa, and Axie by lowest current workload, with `random.Random(42)` used only when the lowest workloads are tied. Only completed CSBS visits with a valid visit month appear in Jessi's master table. Detailed trace, exclusion, mapping, workload, and manifest files remain available for audit but are not part of the simple handoff.


## Data

### 1. Imports and explicit configuration


In [1]:
import json
import logging
import os
import random
import re
import sys
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import urlparse

import pandas as pd
import requests
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

API_URL = "https://redcap.research.sc.edu/api/"
TOKEN_ENV_VAR = "REDCAP_API_TOKEN"
OUTPUT_DIR = Path("csbs_redcap_outputs")
REQUEST_TIMEOUT = (10, 90)
RANDOM_SEED = 42
PREVIEW_ROWS = 20

CLINICIANS = ("Emma", "Tessa", "Axie")
KNOWN_EXAMINER_VALUE_TO_SCORERS = {
    "Emma Platt": ("Emma",),
    "Tessa Djiko": ("Tessa",),
    "Axie Acosta": ("Axie",),
    "Eilis McLaughlin": (),
    "Katie Rowland": (),
    "Alexis Federico": (),
    "Elizabeth Dixon": (),
    "Jessica Bradshaw": (),
    "Axie Acosta & Emma Platt (coscored)": ("Axie", "Emma"),
    "Elizabeth Dixon & Axie Acosta (co-score)": ("Axie",),
    "Emma Platt, Alexis Federico, Jessica Bradshaw (scored during clinical meeting)": ("Emma",),
    "Tessa Djiko, Axie Acosta, Eilis McLaughlin (all coscored)": ("Tessa", "Axie"),
}
COMPOSITE_EXAMINER_VALUES = {
    value for value in KNOWN_EXAMINER_VALUE_TO_SCORERS if any(mark in value for mark in (" & ", ", "))
}
RECORD_ID_FIELD = "demo_id"
CSBS_FORM_NAMES = ("csbs_bs",)
TARGET_VISIT_MONTHS = (9, 12, 15, 18, 24)
DDE_RECORD_ID_SUFFIX_PATTERN = r"--[12]$"
IGNORE_DDE_RECORDS = True
REFERENCE_REPORT_ID = 4692
REFERENCE_REPORT_NAME = "CSBS BS- All Ages, Validated"
EXAMINER_FIELD = "csbsbs_examiner"
EVALUATION_DATE_FIELD = "csbsbs_doe"
COMPLETION_FIELD = "csbs_bs_complete"
COMPLETE_STATUS_VALUE = "2"
EVENT_FIELD_HANDLING = "derive numeric visit month from redcap_event_name"
REPEATING_INSTANCE_HANDLING = "retain repeat instrument and instance when REDCap returns them"
ALLOW_BLANK_EXAMINER_WORKLOAD_FALLBACK = True
JESSI_RECORD_ID = os.getenv("JESSI_RECORD_ID", "1108")
KNOWN_EXAMINER_ASSERTIONS = {
    ("1108", 9): "Emma Platt",
    ("1108", 12): "Alexis Federico",
    ("1108", 15): "Eilis McLaughlin",
    ("1108", 18): "Axie Acosta",
    ("1108", 24): "Eilis McLaughlin",
}
PIPELINE_VERSION = "ipsa-csbs-live-2026-07-20-v5-ignore-double-entry-records"

STANDARD_RULE_LABELS = {
    "NS",
    "NS lower workload",
    "NS random after workload tie",
    "LV",
    "FT",
    "FT lower workload",
    "FT random after workload tie",
}
BLANK_EXAMINER_RULE_LABELS = {
    "Blank examiner lower workload",
    "Blank examiner random after workload tie",
}

LOGGER = logging.getLogger("ipsa_csbs_assignment")
if not LOGGER.handlers:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")


### 2. Read-only REDCap discovery and minimal export

Metadata and event mappings are inspected first. The record request then uses only the configured ID, examiner, Date of Evaluation, and completion fields across CSBS-BS events. Scoring rows include completed base-ID forms at 9, 12, 15, 18, or 24 months and reconcile exactly to the project-native validated report. Any ID ending in `--1` or `--2` is classified as a double-entry validation copy and excluded before contact-history or assignment calculations.


In [2]:
class RedcapAPIError(RuntimeError):
    pass


def _safe_text(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def _normalize_examiner_alias(value):
    return " ".join(unicodedata.normalize("NFKC", _safe_text(value)).split()).casefold()


def _build_known_examiner_lookup():
    lookup = {}
    for raw_value, scorers in KNOWN_EXAMINER_VALUE_TO_SCORERS.items():
        key = _normalize_examiner_alias(raw_value)
        if not key:
            raise ValueError("Known examiner values cannot be blank.")
        invalid = set(scorers).difference(CLINICIANS)
        if invalid:
            raise ValueError(f"Examiner mapping targets unknown scorers: {sorted(invalid)!r}")
        if len(set(scorers)) != len(scorers):
            raise ValueError(f"Examiner mapping repeats a scorer: {raw_value!r}")
        mapped_scorers = tuple(scorers)
        if key in lookup and lookup[key] != mapped_scorers:
            raise ValueError(f"Conflicting normalized examiner value: {raw_value!r}")
        lookup[key] = mapped_scorers
    return lookup


KNOWN_EXAMINER_LOOKUP = _build_known_examiner_lookup()


def _sanitize_redcap_text(value, max_length=400):
    text = " ".join(str(value).replace("\n", " ").split())
    text = re.sub(r"\b[a-fA-F0-9]{24,128}\b", "[redacted]", text)
    text = re.sub(
        r"(?i)(token|api[_ -]?key|authorization)(\s*[:=]\s*)([^\s,;]+)",
        r"\1\2[redacted]",
        text,
    )
    return text[:max_length] + ("…" if len(text) > max_length else "")


def _build_payload(token, content, scalar_params=None, list_params=None):
    payload = [
        ("token", token),
        ("content", content),
        ("format", "json"),
        ("returnFormat", "json"),
    ]
    for key, value in (scalar_params or {}).items():
        if value is not None:
            payload.append((key, str(value)))
    for key, values in (list_params or {}).items():
        for index, value in enumerate(values):
            payload.append((f"{key}[{index}]", str(value)))
    return payload


def redcap_post(token, content, scalar_params=None, list_params=None):
    response = requests.post(
        API_URL,
        data=_build_payload(token, content, scalar_params, list_params),
        timeout=REQUEST_TIMEOUT,
    )
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        body = _sanitize_redcap_text(response.text)
        raise RedcapAPIError(
            f"HTTP {response.status_code} for read-only content={content!r}: {body}"
        ) from exc

    try:
        payload = response.json()
    except ValueError as exc:
        body = _sanitize_redcap_text(response.text)
        raise RedcapAPIError(
            f"Non-JSON response for read-only content={content!r}: {body}"
        ) from exc

    if isinstance(payload, dict) and payload.get("error"):
        raise RedcapAPIError(
            f"REDCap error for read-only content={content!r}: "
            f"{_sanitize_redcap_text(payload['error'])}"
        )
    return payload


def export_discovery_tables(token):
    redcap_post(token, "project")
    metadata_df = pd.DataFrame(redcap_post(token, "metadata"))
    instruments_df = pd.DataFrame(redcap_post(token, "instrument"))
    form_event_mapping_df = pd.DataFrame(redcap_post(token, "formEventMapping"))

    try:
        repeating_df = pd.DataFrame(redcap_post(token, "repeatingFormsEvents"))
        repeating_status = "available"
    except RedcapAPIError:
        repeating_df = pd.DataFrame()
        repeating_status = "endpoint unavailable; record export context used"

    return metadata_df, instruments_df, form_event_mapping_df, repeating_df, repeating_status


def validate_and_get_csbs_events(metadata_df, instruments_df, form_event_mapping_df):
    metadata_fields = set(metadata_df.get("field_name", pd.Series(dtype=str)).astype(str))
    metadata_forms = set(metadata_df.get("form_name", pd.Series(dtype=str)).astype(str))
    instrument_names = set(
        instruments_df.get("instrument_name", pd.Series(dtype=str)).astype(str)
    )

    if RECORD_ID_FIELD not in metadata_fields:
        raise ValueError(f"Configured ID field is absent from metadata: {RECORD_ID_FIELD}")
    if EXAMINER_FIELD not in metadata_fields:
        raise ValueError(
            f"Configured examiner field is absent from metadata: {EXAMINER_FIELD}"
        )
    if EVALUATION_DATE_FIELD not in metadata_fields:
        raise ValueError(
            f"Configured evaluation-date field is absent from metadata: {EVALUATION_DATE_FIELD}"
        )
    examiner_forms = set(
        metadata_df.loc[
            metadata_df["field_name"].astype(str).eq(EXAMINER_FIELD),
            "form_name",
        ].astype(str)
    )
    if examiner_forms != set(CSBS_FORM_NAMES):
        raise ValueError(
            f"Configured examiner field {EXAMINER_FIELD!r} is mapped to "
            f"{sorted(examiner_forms)!r}, not {list(CSBS_FORM_NAMES)!r}."
        )
    date_forms = set(
        metadata_df.loc[
            metadata_df["field_name"].astype(str).eq(EVALUATION_DATE_FIELD),
            "form_name",
        ].astype(str)
    )
    if date_forms != set(CSBS_FORM_NAMES):
        raise ValueError(
            f"Configured evaluation-date field {EVALUATION_DATE_FIELD!r} is mapped to "
            f"{sorted(date_forms)!r}, not {list(CSBS_FORM_NAMES)!r}."
        )
    for form_name in CSBS_FORM_NAMES:
        if form_name not in metadata_forms or form_name not in instrument_names:
            raise ValueError(f"Configured CSBS form is not confirmed by discovery: {form_name}")

    required_mapping_columns = {"form", "unique_event_name"}
    missing_mapping_columns = required_mapping_columns.difference(form_event_mapping_df.columns)
    if missing_mapping_columns:
        raise ValueError(
            "Form-event mapping lacks required columns: "
            + ", ".join(sorted(missing_mapping_columns))
        )

    events = sorted(
        form_event_mapping_df.loc[
            form_event_mapping_df["form"].isin(CSBS_FORM_NAMES),
            "unique_event_name",
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    if not events:
        raise ValueError("No REDCap events are mapped to the configured CSBS form.")

    scoring_events = []
    discovered_target_months = set()
    for event_name in events:
        match = re.fullmatch(r"(\d+)_months?_arm_\d+", event_name)
        if match and int(match.group(1)) in TARGET_VISIT_MONTHS:
            scoring_events.append(event_name)
            discovered_target_months.add(int(match.group(1)))

    missing_target_months = set(TARGET_VISIT_MONTHS).difference(discovered_target_months)
    if missing_target_months:
        raise ValueError(
            "Configured target CSBS months are not mapped to the form: "
            + ", ".join(str(month) for month in sorted(missing_target_months))
        )
    return events, sorted(scoring_events)


def export_minimal_csbs_records(token, csbs_events):
    scalar_params = {
        "type": "flat",
        "rawOrLabel": "raw",
        "rawOrLabelHeaders": "raw",
        "exportCheckboxLabel": "false",
        "exportDataAccessGroups": "false",
    }
    list_params = {
        "fields": [
            RECORD_ID_FIELD,
            EXAMINER_FIELD,
            EVALUATION_DATE_FIELD,
            COMPLETION_FIELD,
        ],
        "events": csbs_events,
    }
    raw_df = pd.DataFrame(redcap_post(token, "record", scalar_params, list_params))

    required_returned = {
        RECORD_ID_FIELD,
        "redcap_event_name",
        EXAMINER_FIELD,
        EVALUATION_DATE_FIELD,
        COMPLETION_FIELD,
    }
    missing_returned = required_returned.difference(raw_df.columns)
    if missing_returned:
        if EXAMINER_FIELD in missing_returned:
            raise RedcapAPIError(
                f"REDCap omitted required field {EXAMINER_FIELD!r}. This is a "
                "field-level export/permission failure, not a blank examiner value. "
                "The token owner needs Full Data Set export rights for the CSBS-BS "
                "instrument before a scoring master can be generated. No outputs were written."
            )
        raise RuntimeError(
            "Minimal record export is missing required columns: "
            + ", ".join(sorted(missing_returned))
        )

    examiner_field_returned = True

    for context_column in ("redcap_repeat_instrument", "redcap_repeat_instance"):
        if context_column not in raw_df.columns:
            raw_df[context_column] = ""

    allowed_columns = [
        RECORD_ID_FIELD,
        "redcap_event_name",
        "redcap_repeat_instrument",
        "redcap_repeat_instance",
        EXAMINER_FIELD,
        EVALUATION_DATE_FIELD,
        COMPLETION_FIELD,
    ]
    raw_df = raw_df.loc[
        raw_df["redcap_event_name"].astype(str).isin(csbs_events), allowed_columns
    ].copy()

    nonblank_examiner_rows = int(
        raw_df[EXAMINER_FIELD].fillna("").astype(str).str.strip().ne("").sum()
    )
    if nonblank_examiner_rows == 0:
        raise RuntimeError(
            f"REDCap returned {EXAMINER_FIELD!r} but every value is blank. "
            "Record 1108 is known from the REDCap UI to contain Emma Platt, so the "
            "export cannot be accepted as complete. No outputs were written."
        )
    return raw_df, examiner_field_returned, nonblank_examiner_rows


def export_reference_report_population(token):
    scalar_params = {
        "report_id": REFERENCE_REPORT_ID,
        "rawOrLabel": "raw",
        "rawOrLabelHeaders": "raw",
        "exportCheckboxLabel": "false",
    }
    report_df = pd.DataFrame(redcap_post(token, "report", scalar_params))
    required = {RECORD_ID_FIELD, "redcap_event_name"}
    missing = required.difference(report_df.columns)
    if missing:
        raise RuntimeError(
            f"Reference report {REFERENCE_REPORT_ID} lacks comparison columns: {sorted(missing)!r}"
        )
    keys = report_df[[RECORD_ID_FIELD, "redcap_event_name"]].copy()
    keys[RECORD_ID_FIELD] = keys[RECORD_ID_FIELD].map(_safe_text)
    keys["redcap_event_name"] = keys["redcap_event_name"].map(_safe_text)
    if keys.eq("").any().any():
        raise RuntimeError(f"Reference report {REFERENCE_REPORT_ID} contains blank comparison keys.")
    return keys


def validate_reference_report_population(eligible_df, report_keys_df):
    if eligible_df["ID"].astype(str).map(_is_dde_record_id).any():
        raise AssertionError("Double-entry IDs reached the report-validation population.")
    reference_scope_df = eligible_df.copy()
    expected = Counter(
        zip(
            reference_scope_df["ID"].astype(str),
            reference_scope_df["redcap_event_name"].astype(str),
        )
    )
    observed = Counter(
        zip(
            report_keys_df[RECORD_ID_FIELD].astype(str),
            report_keys_df["redcap_event_name"].astype(str),
        )
    )
    if expected != observed:
        missing_from_export = int(sum((observed - expected).values()))
        extra_in_export = int(sum((expected - observed).values()))
        raise AssertionError(
            f"Unsuffixed live population differs from REDCap report {REFERENCE_REPORT_ID}: "
            f"{missing_from_export} report rows missing from export; "
            f"{extra_in_export} extra export rows. No outputs were written."
        )
    return {
        "row_count": int(len(report_keys_df)),
        "reference_scope_export_rows": int(len(reference_scope_df)),
        "id_event_multiset_match": True,
    }


def build_field_mapping_audit(
    metadata_df, examiner_field_returned, nonblank_examiner_rows, mapping_counts
):
    columns = ["field_name", "field_label", "form_name", "field_type"]
    available = metadata_df.copy()
    for column in columns:
        if column not in available.columns:
            available[column] = ""
        available[column] = available[column].map(_safe_text)

    audit_df = available.loc[
        available["form_name"].isin(CSBS_FORM_NAMES)
        | available["field_name"].eq(RECORD_ID_FIELD),
        columns,
    ].copy()

    def candidate_role(row):
        field_name = row["field_name"]
        field_label = row["field_label"].casefold()
        if field_name == RECORD_ID_FIELD:
            return "ID"
        if field_name == EXAMINER_FIELD:
            return "examiner"
        if field_name == EVALUATION_DATE_FIELD:
            return "evaluation date"
        if "month" in field_name.casefold() or "month" in field_label or "age" in field_label:
            return "visit month candidate"
        return "CSBS indicator"

    audit_df["candidate_semantic_role"] = audit_df.apply(candidate_role, axis=1)
    audit_df["mapping_decision"] = "not exported (not required)"
    audit_df.loc[audit_df["field_name"].eq(RECORD_ID_FIELD), "mapping_decision"] = (
        "selected base ID; records ending --1/--2 are ignored as double-entry copies"
    )
    audit_df.loc[audit_df["field_name"].eq(EXAMINER_FIELD), "mapping_decision"] = (
        "selected examiner source; exact observed-value allowlist; preserve raw name; row-level blank fallback only"
    )
    audit_df.loc[audit_df["field_name"].eq(EVALUATION_DATE_FIELD), "mapping_decision"] = (
        "selected Date of Evaluation; strict YYYY-MM-DD parsing; used for FT distance in days"
    )
    age_candidate_mask = audit_df["candidate_semantic_role"].eq("visit month candidate")
    audit_df.loc[age_candidate_mask, "mapping_decision"] = (
        "rejected: score age-equivalent is not visit month"
    )
    audit_df["live_export_status"] = "not requested"
    audit_df.loc[audit_df["field_name"].eq(RECORD_ID_FIELD), "live_export_status"] = "returned"
    audit_df.loc[audit_df["field_name"].eq(EVALUATION_DATE_FIELD), "live_export_status"] = "returned"
    audit_df.loc[audit_df["field_name"].eq(EXAMINER_FIELD), "live_export_status"] = (
        f"{'returned' if examiner_field_returned else 'not returned'}; "
        f"{nonblank_examiner_rows} raw nonblank across the full export; "
        f"base assignment population: {mapping_counts['recognized_examiner_rows']} recognized; "
        f"{mapping_counts['recognized_pool_examiner_rows']} name at least one scorer; "
        f"{mapping_counts['recognized_nonpool_examiner_rows']} name only non-pool staff; "
        f"{mapping_counts['recognized_composite_examiner_rows']} composite; "
        f"{mapping_counts['missing_examiner_rows']} blank; "
        f"{mapping_counts['unexpected_examiner_rows']} relevant unexpected; "
        f"{mapping_counts['irrelevant_unexpected_examiner_rows']} irrelevant unexpected"
    )

    system_rows = pd.DataFrame(
        [
            {
                "field_name": "redcap_event_name",
                "field_label": "REDCap unique event name",
                "form_name": "[system]",
                "field_type": "system",
                "candidate_semantic_role": "visit month",
                "mapping_decision": (
                    "selected; numeric month derived from event name; scoring output "
                    f"restricted to months {list(TARGET_VISIT_MONTHS)}"
                ),
                "live_export_status": "returned",
            },
            {
                "field_name": COMPLETION_FIELD,
                "field_label": "REDCap generated CSBS-BS form completion status",
                "form_name": CSBS_FORM_NAMES[0],
                "field_type": "system",
                "candidate_semantic_role": "completion",
                "mapping_decision": f"selected; complete when raw value is {COMPLETE_STATUS_VALUE}",
                "live_export_status": "returned",
            },
        ]
    )
    return (
        pd.concat([audit_df, system_rows], ignore_index=True)
        .sort_values(["form_name", "field_name"], kind="stable")
        .reset_index(drop=True)
    )


## Results

### 3. Normalize eligible visits and assign scorers

The extraction and assignment functions are separate. Candidate history comes from all completed CSBS rows with a recognized examiner, while scoring eligibility is evaluated independently.


In [3]:
MASTER_COLUMNS = [
    "ID",
    "Visit Month",
    "Examiner",
    "Assigned Scoring Clinician",
]

ASSIGNMENT_COLUMNS = [
    "ID",
    "Visit Month",
    "Examiner",
    "Assigned Scoring Clinician",
    "Evaluation Date",
    "Assignment Rule",
    "Assigned Clinician Visit Count",
    "Assigned Clinician Nearest Gap (days)",
    "Workload Before Assignment",
    "redcap_event_name",
    "redcap_repeat_instrument",
    "redcap_repeat_instance",
]

TRACE_COLUMNS = [
    "Assignment Row",
    "ID",
    "Visit Month",
    "Examiner",
    "Evaluation Date",
    "redcap_event_name",
    "redcap_repeat_instrument",
    "redcap_repeat_instance",
    "Candidate",
    "Candidate Visit Count n(c,i)",
    "Never Seen NS",
    "Least Visits LV",
    "Furthest in Time FT",
    "Nearest Gap d(c,r) (days)",
    "Workload Before W(c,t)",
    "Selected",
    "Winning Rule",
]

EXCLUSION_COLUMNS = [
    "Audit Type",
    "ID",
    "Visit Month",
    "Examiner",
    "Raw Examiner Value",
    "Examiner Mapping Status",
    "redcap_event_name",
    "redcap_repeat_instrument",
    "redcap_repeat_instance",
    "CSBS Completion Status",
    "Exclusion Reason",
    "Exclusion Detail",
]


def _derive_visit_month(event_name):
    match = re.fullmatch(r"(\d+)_months?_arm_\d+", _safe_text(event_name))
    return int(match.group(1)) if match else pd.NA


def _is_complete(value):
    return _safe_text(value) == COMPLETE_STATUS_VALUE


def _parse_evaluation_date(value):
    text = _safe_text(value)
    if not text:
        return pd.NaT
    return pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")


def _parse_examiner(value):
    text = _safe_text(value)
    normalized = _normalize_examiner_alias(text)
    if not normalized or normalized in {"na", "n/a"}:
        return pd.NA, (), "missing"
    scorers = KNOWN_EXAMINER_LOOKUP.get(normalized)
    if scorers is None:
        return text, (), "unexpected"
    return text, scorers, "recognized"


def _record_id_as_exported(value):
    if value is None or pd.isna(value):
        return ""
    return str(value)


def _is_included_record_id(record_id):
    has_id = bool(str(record_id).strip())
    return has_id and not (IGNORE_DDE_RECORDS and _is_dde_record_id(record_id))


def _is_dde_record_id(record_id):
    return bool(re.search(DDE_RECORD_ID_SUFFIX_PATTERN, str(record_id)))


def normalize_csbs_records(raw_df, csbs_events, scoring_events):
    eligible_rows = []
    contact_rows = []
    audit_rows = []
    completed_rows = 0
    included_completed_contact_rows = 0
    included_completed_scoring_rows = 0
    included_completed_missing_evaluation_dates = 0
    relevant_unexpected_examiner_rows = 0
    irrelevant_unexpected_examiner_rows = 0

    for source_row, row in raw_df.reset_index(drop=True).iterrows():
        record_id = _record_id_as_exported(row.get(RECORD_ID_FIELD, pd.NA))
        event_name = _safe_text(row.get("redcap_event_name", pd.NA))
        repeat_instrument = _safe_text(row.get("redcap_repeat_instrument", pd.NA))
        repeat_instance = _safe_text(row.get("redcap_repeat_instance", pd.NA))
        completion_status = _safe_text(row.get(COMPLETION_FIELD, pd.NA))
        visit_month = _derive_visit_month(event_name)
        raw_examiner_value = _safe_text(row.get(EXAMINER_FIELD, pd.NA))
        evaluation_date = _parse_evaluation_date(
            row.get(EVALUATION_DATE_FIELD, pd.NA)
        )
        examiner, examiner_scorers, examiner_status = _parse_examiner(
            raw_examiner_value
        )
        is_csbs = event_name in csbs_events
        is_scoring_event = event_name in scoring_events
        is_complete = _is_complete(completion_status)
        is_included_record = _is_included_record_id(record_id)
        is_dde_record = _is_dde_record_id(record_id)
        if examiner_status == "unexpected":
            if is_csbs and is_complete and is_included_record:
                relevant_unexpected_examiner_rows += 1
            else:
                irrelevant_unexpected_examiner_rows += 1

        context = {
            "ID": record_id,
            "Visit Month": visit_month,
            "Examiner": examiner,
            "Raw Examiner Value": raw_examiner_value,
            "Examiner Mapping Status": examiner_status,
            "redcap_event_name": event_name,
            "redcap_repeat_instrument": repeat_instrument,
            "redcap_repeat_instance": repeat_instance,
            "CSBS Completion Status": completion_status,
        }

        if is_csbs and is_complete:
            completed_rows += 1
            if is_included_record:
                included_completed_contact_rows += 1
                if pd.isna(evaluation_date):
                    included_completed_missing_evaluation_dates += 1
                if is_scoring_event:
                    included_completed_scoring_rows += 1
            if is_included_record and examiner_status == "recognized":
                for scoring_clinician in examiner_scorers:
                    contact_rows.append(
                        {
                            "ID": record_id,
                            "Visit Month": visit_month,
                            "Evaluation Date": evaluation_date,
                            "Examiner": scoring_clinician,
                        }
                    )

        exclusion_reason = None
        exclusion_detail = ""
        if not is_csbs:
            exclusion_reason = "not a CSBS form/event"
        elif not str(record_id).strip():
            exclusion_reason = "missing child ID"
        elif is_dde_record:
            exclusion_reason = "double-entry validation record ignored"
            exclusion_detail = (
                "Record ID ends in --1 or --2 and is an automatically generated "
                "double-entry copy; it is excluded from assignments and contact history."
            )
        elif not is_scoring_event:
            exclusion_reason = "not a targeted CSBS scoring event"
            exclusion_detail = (
                f"Jessi-facing scoring months are {list(TARGET_VISIT_MONTHS)}. "
                "Other mapped CSBS events remain available only for contact history."
            )
        elif not is_complete:
            exclusion_reason = "incomplete CSBS form"
        elif pd.isna(visit_month):
            exclusion_reason = "missing visit month"
            exclusion_detail = "No numeric month can be derived from the mapped event name."
        elif examiner_status == "unexpected":
            exclusion_reason = "unexpected examiner value"
            exclusion_detail = "Value is not in the exact live examiner allowlist."
        elif examiner_status == "missing" and not ALLOW_BLANK_EXAMINER_WORKLOAD_FALLBACK:
            exclusion_reason = "missing examiner"

        if exclusion_reason:
            audit_rows.append(
                {
                    "Audit Type": "Exclusion",
                    **context,
                    "Exclusion Reason": exclusion_reason,
                    "Exclusion Detail": exclusion_detail,
                }
            )
            continue

        eligible_rows.append(
            {
                "_source_row": int(source_row),
                "_raw_record_id": record_id,
                "ID": record_id,
                "Visit Month": int(visit_month),
                "Examiner": examiner,
                "_raw_examiner_value": raw_examiner_value,
                "_examiner_scorers": tuple(examiner_scorers),
                "_examiner_mapping_status": examiner_status,
                "_evaluation_date": evaluation_date,
                "redcap_event_name": event_name,
                "redcap_repeat_instrument": repeat_instrument,
                "redcap_repeat_instance": repeat_instance,
                "_is_csbs": bool(is_csbs),
                "_is_target_scoring_event": bool(is_scoring_event),
                "_is_included_record": bool(is_included_record),
                "_is_dde_record": bool(is_dde_record),
                "_is_complete": bool(is_complete),
                "_used_blank_examiner_fallback": examiner_status == "missing",
            }
        )

    if included_completed_missing_evaluation_dates:
        raise RuntimeError(
            f"{included_completed_missing_evaluation_dates} completed CSBS rows with included base IDs "
            f"lack a valid {EVALUATION_DATE_FIELD} value. FT distance cannot be "
            "computed without inventing time data. No outputs were written."
        )

    eligible_columns = [
        "_source_row",
        "_raw_record_id",
        "ID",
        "Visit Month",
        "Examiner",
        "_raw_examiner_value",
        "_examiner_scorers",
        "_examiner_mapping_status",
        "_evaluation_date",
        "redcap_event_name",
        "redcap_repeat_instrument",
        "redcap_repeat_instance",
        "_is_csbs",
        "_is_target_scoring_event",
        "_is_included_record",
        "_is_dde_record",
        "_is_complete",
        "_used_blank_examiner_fallback",
    ]
    eligible_df = pd.DataFrame(eligible_rows, columns=eligible_columns)
    raw_pool_contact_mentions = int(len(contact_rows))
    contact_history_df = pd.DataFrame(
        contact_rows, columns=["ID", "Visit Month", "Evaluation Date", "Examiner"]
    )
    if not contact_history_df.empty:
        contact_history_df = (
            contact_history_df.sort_values(
                ["ID", "Examiner", "Evaluation Date", "Visit Month"],
                kind="stable",
                na_position="last",
            )
            .drop_duplicates(["ID", "Examiner", "Evaluation Date"], keep="first")
            .reset_index(drop=True)
        )
    duplicate_contact_mentions_removed = raw_pool_contact_mentions - int(
        len(contact_history_df)
    )
    audit_df = pd.DataFrame(audit_rows, columns=EXCLUSION_COLUMNS)

    duplicate_rows = 0
    if not eligible_df.empty:
        duplicate_mask = eligible_df.duplicated(["ID", "Visit Month"], keep=False)
        duplicate_rows = int(duplicate_mask.sum())
        if duplicate_rows:
            warnings = eligible_df.loc[duplicate_mask].copy()
            warnings["Audit Type"] = "Warning"
            warnings["Raw Examiner Value"] = warnings["_raw_examiner_value"]
            warnings["Examiner Mapping Status"] = warnings["_examiner_mapping_status"]
            warnings["CSBS Completion Status"] = COMPLETE_STATUS_VALUE
            warnings["Exclusion Reason"] = "duplicate visit requiring review"
            warnings["Exclusion Detail"] = (
                "Retained as a distinct visit and ordered by event/repeat context."
            )
            audit_df = pd.concat(
                [audit_df, warnings.reindex(columns=EXCLUSION_COLUMNS)],
                ignore_index=True,
            )

    excluded_rows = int(audit_df["Audit Type"].eq("Exclusion").sum())
    warning_rows = int(audit_df["Audit Type"].eq("Warning").sum())
    counts = {
        "exported_rows": int(len(raw_df)),
        "csbs_filtered_rows": int(len(raw_df)),
        "completed_rows": int(completed_rows),
        "included_completed_contact_rows": int(included_completed_contact_rows),
        "included_completed_scoring_rows": int(included_completed_scoring_rows),
        "included_completed_missing_evaluation_dates": int(
            included_completed_missing_evaluation_dates
        ),
        "raw_pool_contact_mentions": raw_pool_contact_mentions,
        "deduplicated_pool_contacts": int(len(contact_history_df)),
        "duplicate_contact_mentions_removed": int(duplicate_contact_mentions_removed),
        "eligible_rows": int(len(eligible_df)),
        "excluded_rows": excluded_rows,
        "audit_warning_rows": warning_rows,
        "duplicate_rows_flagged": duplicate_rows,
        "exact_record_ids_included": int(eligible_df["ID"].nunique()),
        "base_scoring_rows_included": int((~eligible_df["_is_dde_record"]).sum()),
        "dde_rows_ignored": int(
            audit_df["Exclusion Reason"].eq("double-entry validation record ignored").sum()
        ),
        "completed_target_dde_rows_ignored": int(
            (
                audit_df["Exclusion Reason"].eq("double-entry validation record ignored")
                & audit_df["CSBS Completion Status"].astype(str).eq(COMPLETE_STATUS_VALUE)
                & audit_df["Visit Month"].isin(TARGET_VISIT_MONTHS)
            ).sum()
        ),
        "dde_suffix_1_completed_target_rows_ignored": int(
            (
                audit_df["Exclusion Reason"].eq("double-entry validation record ignored")
                & audit_df["CSBS Completion Status"].astype(str).eq(COMPLETE_STATUS_VALUE)
                & audit_df["Visit Month"].isin(TARGET_VISIT_MONTHS)
                & audit_df["ID"].astype(str).str.endswith("--1")
            ).sum()
        ),
        "dde_suffix_2_completed_target_rows_ignored": int(
            (
                audit_df["Exclusion Reason"].eq("double-entry validation record ignored")
                & audit_df["CSBS Completion Status"].astype(str).eq(COMPLETE_STATUS_VALUE)
                & audit_df["Visit Month"].isin(TARGET_VISIT_MONTHS)
                & audit_df["ID"].astype(str).str.endswith("--2")
            ).sum()
        ),
        "nontarget_csbs_event_rows_excluded": int(
            audit_df["Exclusion Reason"]
            .eq("not a targeted CSBS scoring event")
            .sum()
        ),
        "raw_nonblank_examiner_rows": int(
            raw_df[EXAMINER_FIELD].fillna("").astype(str).str.strip().ne("").sum()
        ),
        "recognized_examiner_rows": int(
            eligible_df["_examiner_mapping_status"].eq("recognized").sum()
        ),
        "recognized_pool_examiner_rows": int(
            eligible_df["_examiner_scorers"].map(bool).sum()
        ),
        "recognized_nonpool_examiner_rows": int(
            (
                eligible_df["_examiner_mapping_status"].eq("recognized")
                & ~eligible_df["_examiner_scorers"].map(bool)
            ).sum()
        ),
        "recognized_composite_examiner_rows": int(
            eligible_df["Examiner"].fillna("").isin(COMPOSITE_EXAMINER_VALUES).sum()
        ),
        "missing_examiner_rows": int(
            eligible_df["_examiner_mapping_status"].eq("missing").sum()
        ),
        "unexpected_examiner_rows": int(relevant_unexpected_examiner_rows),
        "irrelevant_unexpected_examiner_rows": int(
            irrelevant_unexpected_examiner_rows
        ),
        "known_examiner_eligible_rows": int(eligible_df["Examiner"].notna().sum()),
        "blank_examiner_fallback_rows": int(
            eligible_df["_used_blank_examiner_fallback"].sum()
        ),
    }
    return {
        "eligible": eligible_df,
        "contact_history": contact_history_df,
        "audit": audit_df,
        "counts": counts,
    }


def _nearest_gap_days(evaluation_date, evaluation_dates):
    if not evaluation_dates:
        return pd.NA
    return min(abs((evaluation_date - visited_date).days) for visited_date in evaluation_dates)


def _choose_by_workload(candidates, workload, rng):
    minimum = min(workload[clinician] for clinician in candidates)
    finalists = sorted(
        clinician for clinician in candidates if workload[clinician] == minimum
    )
    if len(finalists) == 1:
        return finalists[0], False
    return rng.choice(finalists), True


def assign_scoring_clinicians(scoring_df, contact_history_df, seed=RANDOM_SEED):
    required_internal = {"_examiner_scorers", "_examiner_mapping_status"}
    missing_internal = required_internal.difference(scoring_df.columns)
    if missing_internal:
        raise ValueError(f"Assignment input lacks examiner parsing columns: {sorted(missing_internal)!r}")
    invalid_scoring_examiners = {
        scorer
        for scorers in scoring_df["_examiner_scorers"]
        for scorer in scorers
        if scorer not in CLINICIANS
    }
    invalid_contact_examiners = set(contact_history_df["Examiner"].dropna()).difference(CLINICIANS)
    if invalid_scoring_examiners or invalid_contact_examiners:
        raise ValueError(
            "Assignment input contains non-canonical examiner values: "
            f"{sorted(invalid_scoring_examiners | invalid_contact_examiners)!r}"
        )

    if scoring_df.empty:
        return (
            pd.DataFrame(columns=ASSIGNMENT_COLUMNS),
            pd.DataFrame(columns=TRACE_COLUMNS),
            pd.DataFrame({"Clinician": CLINICIANS, "Assignments": [0, 0, 0]}),
        )

    ordered = scoring_df.copy()
    ordered["_repeat_instance_numeric"] = pd.to_numeric(
        ordered["redcap_repeat_instance"], errors="coerce"
    ).fillna(float("inf"))
    ordered["_repeat_instance_text"] = (
        ordered["redcap_repeat_instance"].fillna("").astype(str)
    )
    ordered = ordered.sort_values(
        [
            "ID",
            "Visit Month",
            "redcap_event_name",
            "redcap_repeat_instrument",
            "_repeat_instance_numeric",
            "_repeat_instance_text",
            "_source_row",
        ],
        kind="stable",
    ).reset_index(drop=True)

    contact_counts = Counter()
    contact_dates = defaultdict(list)
    for _, history_row in contact_history_df.iterrows():
        child_id = str(history_row["ID"])
        clinician = history_row["Examiner"]
        contact_counts[(child_id, clinician)] += 1
        contact_dates[(child_id, clinician)].append(history_row["Evaluation Date"])

    for key in contact_dates:
        contact_dates[key].sort()

    workload = Counter({clinician: 0 for clinician in CLINICIANS})
    rng = random.Random(seed)
    assignment_rows = []
    trace_rows = []

    for assignment_row, row in ordered.iterrows():
        child_id = str(row["ID"])
        month = float(row["Visit Month"])
        evaluation_date = row["_evaluation_date"]
        examiner = row["Examiner"]
        examiner_scorers = set(row["_examiner_scorers"])
        examiner_is_blank = row["_examiner_mapping_status"] == "missing"
        candidates = [
            clinician for clinician in CLINICIANS if clinician not in examiner_scorers
        ]
        if not candidates:
            raise RuntimeError("No eligible scorer remains after examiner exclusion.")

        candidate_counts = {
            clinician: int(contact_counts[(child_id, clinician)])
            for clinician in candidates
        }
        candidate_gaps = {
            clinician: _nearest_gap_days(
                evaluation_date, contact_dates.get((child_id, clinician), [])
            )
            for clinician in candidates
        }

        never_seen = set()
        least_visits = set()
        furthest_in_time = set()

        if examiner_is_blank:
            chosen, used_random = _choose_by_workload(candidates, workload, rng)
            rule = (
                "Blank examiner random after workload tie"
                if used_random
                else "Blank examiner lower workload"
            )
        else:
            never_seen = {
                clinician
                for clinician, count in candidate_counts.items()
                if count == 0
            }
            if never_seen:
                if len(never_seen) == 1:
                    chosen = next(iter(never_seen))
                    rule = "NS"
                else:
                    chosen, used_random = _choose_by_workload(
                        sorted(never_seen), workload, rng
                    )
                    rule = (
                        "NS random after workload tie"
                        if used_random
                        else "NS lower workload"
                    )
            else:
                minimum_count = min(candidate_counts.values())
                least_visits = {
                    clinician
                    for clinician, count in candidate_counts.items()
                    if count == minimum_count
                }
                if len(least_visits) == 1:
                    chosen = next(iter(least_visits))
                    rule = "LV"
                else:
                    if any(pd.isna(candidate_gaps[clinician]) for clinician in least_visits):
                        raise RuntimeError(
                            "FT comparison is undecidable because a tied candidate has contact rows but no valid evaluation date."
                        )
                    largest_gap = max(
                        candidate_gaps[clinician] for clinician in least_visits
                    )
                    furthest_in_time = {
                        clinician
                        for clinician in least_visits
                        if candidate_gaps[clinician] == largest_gap
                    }
                    if len(furthest_in_time) == 1:
                        chosen = next(iter(furthest_in_time))
                        rule = "FT"
                    else:
                        chosen, used_random = _choose_by_workload(
                            sorted(furthest_in_time), workload, rng
                        )
                        rule = (
                            "FT random after workload tie"
                            if used_random
                            else "FT lower workload"
                        )

        reported_candidate_counts = (
            {clinician: pd.NA for clinician in candidates}
            if examiner_is_blank
            else candidate_counts
        )
        reported_candidate_gaps = (
            {clinician: pd.NA for clinician in candidates}
            if examiner_is_blank
            else candidate_gaps
        )
        workload_before = int(workload[chosen])
        assignment_rows.append(
            {
                "ID": child_id,
                "Visit Month": int(month),
                "Examiner": examiner,
                "Assigned Scoring Clinician": chosen,
                "Evaluation Date": evaluation_date.strftime("%Y-%m-%d"),
                "Assignment Rule": rule,
                "Assigned Clinician Visit Count": reported_candidate_counts[chosen],
                "Assigned Clinician Nearest Gap (days)": reported_candidate_gaps[chosen],
                "Workload Before Assignment": workload_before,
                "redcap_event_name": row["redcap_event_name"],
                "redcap_repeat_instrument": row["redcap_repeat_instrument"],
                "redcap_repeat_instance": row["redcap_repeat_instance"],
            }
        )

        for candidate in candidates:
            trace_rows.append(
                {
                    "Assignment Row": int(assignment_row),
                    "ID": child_id,
                    "Visit Month": int(month),
                    "Examiner": examiner,
                    "Evaluation Date": evaluation_date.strftime("%Y-%m-%d"),
                    "redcap_event_name": row["redcap_event_name"],
                    "redcap_repeat_instrument": row["redcap_repeat_instrument"],
                    "redcap_repeat_instance": row["redcap_repeat_instance"],
                    "Candidate": candidate,
                    "Candidate Visit Count n(c,i)": reported_candidate_counts[candidate],
                    "Never Seen NS": pd.NA if examiner_is_blank else candidate in never_seen,
                    "Least Visits LV": pd.NA if examiner_is_blank else candidate in least_visits,
                    "Furthest in Time FT": pd.NA if examiner_is_blank else candidate in furthest_in_time,
                    "Nearest Gap d(c,r) (days)": reported_candidate_gaps[candidate],
                    "Workload Before W(c,t)": int(workload[candidate]),
                    "Selected": candidate == chosen,
                    "Winning Rule": rule,
                }
            )
        workload[chosen] += 1

    assignments_df = pd.DataFrame(assignment_rows, columns=ASSIGNMENT_COLUMNS)
    trace_df = pd.DataFrame(trace_rows, columns=TRACE_COLUMNS)
    workload_df = pd.DataFrame(
        {
            "Clinician": list(CLINICIANS),
            "Assignments": [int(workload[c]) for c in CLINICIANS],
        }
    )
    return assignments_df, trace_df, workload_df


def independently_verify_trace_hierarchy(assignments_df, trace_df, contact_history_df):
    if assignments_df.empty:
        return trace_df.empty

    history_ids = contact_history_df["ID"].astype(str)
    reconstructed_workload = Counter({clinician: 0 for clinician in CLINICIANS})
    for assignment_row, assignment in assignments_df.reset_index(drop=True).iterrows():
        group = trace_df.loc[trace_df["Assignment Row"].eq(assignment_row)].copy()
        selected = group.loc[group["Selected"], "Candidate"].tolist()
        chosen = assignment["Assigned Scoring Clinician"]
        rule = assignment["Assignment Rule"]
        if selected != [chosen] or set(group["Winning Rule"]) != {rule}:
            return False

        workloads = {
            row["Candidate"]: int(row["Workload Before W(c,t)"])
            for _, row in group.iterrows()
        }
        if any(
            workloads.get(candidate) != reconstructed_workload[candidate]
            for candidate in workloads
        ):
            return False
        examiner_is_blank = pd.isna(assignment["Examiner"])
        if examiner_is_blank:
            minimum_workload = min(workloads.values())
            finalists = {c for c, value in workloads.items() if value == minimum_workload}
            expected_rule = (
                "Blank examiner lower workload"
                if len(finalists) == 1
                else "Blank examiner random after workload tie"
            )
        else:
            child_history = contact_history_df.loc[
                history_ids.eq(str(assignment["ID"]))
            ]
            counts = {}
            gaps = {}
            for _, trace_row in group.iterrows():
                candidate = trace_row["Candidate"]
                candidate_history = child_history.loc[
                    child_history["Examiner"].eq(candidate), "Evaluation Date"
                ]
                expected_count = int(len(candidate_history))
                actual_count = int(trace_row["Candidate Visit Count n(c,i)"])
                if actual_count != expected_count:
                    return False
                counts[candidate] = expected_count

                actual_gap = trace_row["Nearest Gap d(c,r) (days)"]
                if expected_count == 0 or candidate_history.isna().any():
                    if not pd.isna(actual_gap):
                        return False
                    gaps[candidate] = pd.NA
                else:
                    assignment_date = pd.Timestamp(assignment["Evaluation Date"])
                    expected_gap = min(
                        abs((assignment_date - pd.Timestamp(contact_date)).days)
                        for contact_date in candidate_history
                    )
                    if pd.isna(actual_gap) or float(actual_gap) != expected_gap:
                        return False
                    gaps[candidate] = expected_gap

            minimum_count = min(counts.values())
            finalists = {c for c, value in counts.items() if value == minimum_count}
            if len(finalists) == 1:
                expected_rule = "NS" if minimum_count == 0 else "LV"
            elif minimum_count == 0:
                minimum_workload = min(workloads[c] for c in finalists)
                finalists = {c for c in finalists if workloads[c] == minimum_workload}
                expected_rule = (
                    "NS lower workload"
                    if len(finalists) == 1
                    else "NS random after workload tie"
                )
            else:
                if any(pd.isna(gaps[c]) for c in finalists):
                    return False
                largest_gap = max(gaps[c] for c in finalists)
                finalists = {c for c in finalists if gaps[c] == largest_gap}
                if len(finalists) == 1:
                    expected_rule = "FT"
                else:
                    minimum_workload = min(workloads[c] for c in finalists)
                    finalists = {c for c in finalists if workloads[c] == minimum_workload}
                    expected_rule = (
                        "FT lower workload"
                        if len(finalists) == 1
                        else "FT random after workload tie"
                    )

        if chosen not in finalists or rule != expected_rule:
            return False
        reconstructed_workload[chosen] += 1
    return True


def run_quality_checks(
    normalized,
    master_df,
    assignments_df,
    trace_df,
    workload_df,
    replay_assignments_df,
    replay_trace_df,
    examiner_field_returned,
    reference_report_validation,
):
    eligible_df = normalized["eligible"]
    contact_history_df = normalized["contact_history"]
    audit_df = normalized["audit"]
    counts = normalized["counts"]
    parsed_assignments = [
        _parse_examiner(value) for value in assignments_df["Examiner"].tolist()
    ]
    assignment_statuses = [parsed[2] for parsed in parsed_assignments]
    assignment_scorer_sets = [set(parsed[1]) for parsed in parsed_assignments]
    expected_candidate_sets = [
        set(CLINICIANS).difference(examiner_scorers)
        for examiner_scorers in assignment_scorer_sets
    ]
    unexpected_audit = audit_df["Examiner Mapping Status"].eq("unexpected")
    unexpected_raw_preserved = bool(
        not unexpected_audit.any()
        or audit_df.loc[unexpected_audit, "Raw Examiner Value"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .all()
    )
    known_assertions_ok = all(
        (
            eligible_df["ID"].astype(str).eq(str(record_id))
            & eligible_df["Visit Month"].eq(int(visit_month))
            & eligible_df["Examiner"].eq(expected_examiner)
        ).any()
        for (record_id, visit_month), expected_examiner in KNOWN_EXAMINER_ASSERTIONS.items()
    )
    expected_trace_rows = int(sum(len(candidates) for candidates in expected_candidate_sets))
    eligible_id_visit_multiset = Counter(
        zip(eligible_df["ID"].astype(str), eligible_df["Visit Month"].astype(int))
    )
    master_id_visit_multiset = Counter(
        zip(master_df["ID"].astype(str), master_df["Visit Month"].astype(int))
    )

    if trace_df.empty:
        trace_group_sizes_ok = assignments_df.empty
        one_selected_per_visit = assignments_df.empty
        selected_matches_assignment = assignments_df.empty
        candidate_pools_match_examiner_exclusions = assignments_df.empty
    else:
        group_sizes = trace_df.groupby("Assignment Row", sort=False).size()
        expected_group_sizes = pd.Series(
            [len(candidates) for candidates in expected_candidate_sets],
            index=range(len(assignments_df)),
            dtype="int64",
        )
        trace_group_sizes_ok = group_sizes.reindex(expected_group_sizes.index).equals(
            expected_group_sizes
        )
        selected_counts = trace_df.groupby("Assignment Row", sort=False)["Selected"].sum()
        one_selected_per_visit = bool(selected_counts.eq(1).all())
        selected = (
            trace_df.loc[trace_df["Selected"], ["Assignment Row", "Candidate"]]
            .set_index("Assignment Row")["Candidate"]
            .sort_index()
        )
        expected_selected = pd.Series(
            assignments_df["Assigned Scoring Clinician"].tolist(),
            index=range(len(assignments_df)),
        )
        selected_matches_assignment = selected.equals(expected_selected)
        candidate_pools_match_examiner_exclusions = all(
            set(
                trace_df.loc[
                    trace_df["Assignment Row"].eq(assignment_row), "Candidate"
                ]
            )
            == expected_candidates
            for assignment_row, expected_candidates in enumerate(expected_candidate_sets)
        )

    duplicate_warning_rows = int(
        (
            audit_df["Audit Type"].eq("Warning")
            & audit_df["Exclusion Reason"].eq("duplicate visit requiring review")
        ).sum()
    )
    allowed_rules = STANDARD_RULE_LABELS | BLANK_EXAMINER_RULE_LABELS
    checks = {
        "required_examiner_field_was_returned": bool(examiner_field_returned),
        "base_population_matches_reference_report_4692": bool(
            reference_report_validation["id_event_multiset_match"]
        ),
        "recognized_examiner_check_is_nonvacuous": int(
            counts["recognized_examiner_rows"]
        ) > 0,
        "record_1108_matches_live_examiner_values_by_visit": bool(known_assertions_ok),
        "jessi_master_has_exact_four_columns": list(master_df.columns) == MASTER_COLUMNS,
        "jessi_master_row_count_equals_assignments": int(len(master_df)) == int(len(assignments_df)),
        "jessi_master_matches_assignment_projection": master_df.reset_index(drop=True).equals(
            assignments_df[MASTER_COLUMNS].reset_index(drop=True)
        ),
        "master_id_visit_multiset_matches_live_base_population": (
            master_id_visit_multiset == eligible_id_visit_multiset
        ),
        "every_included_row_is_csbs": bool(eligible_df.empty or eligible_df["_is_csbs"].all()),
        "every_included_row_is_a_target_scoring_event": bool(
            eligible_df.empty or eligible_df["_is_target_scoring_event"].all()
        ),
        "every_included_row_has_a_nonblank_record_id": bool(
            eligible_df.empty or eligible_df["_is_included_record"].all()
        ),
        "redcap_record_ids_are_preserved_exactly": bool(
            eligible_df["ID"].equals(eligible_df["_raw_record_id"])
        ),
        "no_double_entry_ids_appear_in_master": bool(
            master_df.empty
            or not master_df["ID"].astype(str).map(_is_dde_record_id).any()
        ),
        "completed_double_entry_rows_are_explicitly_ignored": bool(
            counts["completed_target_dde_rows_ignored"] > 0
            and counts["dde_suffix_1_completed_target_rows_ignored"] > 0
            and counts["dde_suffix_2_completed_target_rows_ignored"] > 0
            and counts["completed_target_dde_rows_ignored"]
            == counts["dde_suffix_1_completed_target_rows_ignored"]
            + counts["dde_suffix_2_completed_target_rows_ignored"]
        ),
        "master_has_no_duplicate_exact_id_visit_keys": bool(
            master_df.empty or not master_df.duplicated(["ID", "Visit Month"]).any()
        ),
        "contact_history_uses_only_nonblank_exact_record_ids": bool(
            contact_history_df.empty
            or (
                contact_history_df["ID"].astype(str).map(_is_included_record_id).all()
                and not contact_history_df["ID"].astype(str).map(_is_dde_record_id).any()
            )
        ),
        "contact_history_has_valid_evaluation_dates": bool(
            contact_history_df.empty
            or contact_history_df["Evaluation Date"].notna().all()
        ),
        "contact_history_deduplicates_same_child_clinician_date": bool(
            contact_history_df.empty
            or not contact_history_df.duplicated(
                ["ID", "Examiner", "Evaluation Date"]
            ).any()
        ),
        "every_included_row_is_complete": bool(eligible_df.empty or eligible_df["_is_complete"].all()),
        "every_included_row_has_valid_evaluation_date": bool(
            eligible_df.empty or eligible_df["_evaluation_date"].notna().all()
        ),
        "every_included_row_has_valid_numeric_visit_month": bool(
            assignments_df.empty
            or pd.to_numeric(assignments_df["Visit Month"], errors="coerce").notna().all()
        ),
        "every_included_visit_month_matches_reference_report": bool(
            assignments_df.empty
            or assignments_df["Visit Month"].isin(TARGET_VISIT_MONTHS).all()
        ),
        "every_nonmissing_examiner_is_recognized": all(
            status in {"recognized", "missing"} for status in assignment_statuses
        ),
        "master_examiner_is_exact_live_value_or_blank": all(
            status in {"recognized", "missing"}
            for _, _, status in (
                _parse_examiner(value) for value in master_df["Examiner"].tolist()
            )
        ),
        "no_relevant_unexpected_examiner_values": int(
            counts["unexpected_examiner_rows"]
        ) == 0,
        "unexpected_raw_examiner_values_are_preserved_in_audit": unexpected_raw_preserved,
        "blank_fallback_is_only_used_for_true_missing_cells": bool(
            eligible_df.empty
            or eligible_df.loc[
                eligible_df["_used_blank_examiner_fallback"],
                "_examiner_mapping_status",
            ].eq("missing").all()
        ),
        "no_assigned_scorer_is_named_in_examiner_value": all(
            assigned not in examiner_scorers
            for assigned, examiner_scorers in zip(
                assignments_df["Assigned Scoring Clinician"],
                assignment_scorer_sets,
            )
        ),
        "every_included_row_receives_exactly_one_scorer": bool(
            assignments_df.empty
            or assignments_df["Assigned Scoring Clinician"].isin(CLINICIANS).all()
        ),
        "assignment_count_equals_number_of_included_visits": int(len(assignments_df))
        == int(counts["eligible_rows"]),
        "eligible_count_matches_all_completed_scoring_records": int(
            counts["eligible_rows"]
        ) == int(counts["included_completed_scoring_rows"]),
        "workload_total_equals_number_of_included_visits": int(workload_df["Assignments"].sum())
        == int(len(assignments_df)),
        "workload_summary_contains_all_clinicians": set(workload_df["Clinician"])
        == set(CLINICIANS),
        "candidate_trace_row_count_matches_candidate_pools": int(len(trace_df))
        == expected_trace_rows,
        "candidate_trace_group_sizes_are_correct": bool(trace_group_sizes_ok),
        "candidate_pools_exactly_exclude_named_scoring_clinicians": bool(
            candidate_pools_match_examiner_exclusions
        ),
        "candidate_trace_has_one_selected_candidate_per_visit": bool(one_selected_per_visit),
        "candidate_trace_selected_candidate_matches_assignment": bool(
            selected_matches_assignment
        ),
        "independent_trace_hierarchy_and_history_verification": bool(
            independently_verify_trace_hierarchy(
                assignments_df, trace_df, contact_history_df
            )
        ),
        "assignment_rule_labels_are_valid": set(assignments_df["Assignment Rule"]).issubset(
            allowed_rules
        ),
        "duplicate_rows_are_flagged_for_review": duplicate_warning_rows
        == int(counts["duplicate_rows_flagged"]),
        "every_exported_row_has_one_disposition": int(counts["eligible_rows"])
        + int(counts["excluded_rows"])
        == int(counts["csbs_filtered_rows"]),
        "fixed_seed_replay_matches_assignments": assignments_df.equals(
            replay_assignments_df
        ),
        "fixed_seed_replay_matches_candidate_trace": trace_df.equals(replay_trace_df),
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise AssertionError("Quality-control checks failed: " + ", ".join(failed))
    return checks


def run_jessi_validation(assignments_df, trace_df, record_id=JESSI_RECORD_ID):
    if not record_id:
        print("Jessi validation not run: set JESSI_RECORD_ID to a live IPSA record ID.")
        return {"status": "not_run", "reason": "record ID not supplied"}

    record_id = str(record_id)
    jessi_rows = assignments_df.loc[
        assignments_df["ID"].astype(str).eq(record_id),
        [
            "ID",
            "Visit Month",
            "Examiner",
            "Assigned Scoring Clinician",
            "Assignment Rule",
            "Assigned Clinician Visit Count",
            "Assigned Clinician Nearest Gap (days)",
            "Workload Before Assignment",
        ],
    ].reset_index(drop=True)
    jessi_trace = trace_df.loc[
        trace_df["ID"].astype(str).eq(record_id),
        [
            "ID",
            "Visit Month",
            "Examiner",
            "Candidate",
            "Candidate Visit Count n(c,i)",
            "Never Seen NS",
            "Least Visits LV",
            "Furthest in Time FT",
            "Nearest Gap d(c,r) (days)",
            "Workload Before W(c,t)",
            "Selected",
            "Winning Rule",
        ],
    ].reset_index(drop=True)
    display(jessi_rows)
    display(jessi_trace)
    print(
        "Jessi rows and candidate trace shown. No pass/fail claim is made because "
        "no expected live assignment table was supplied."
    )
    return {"status": "trace_only", "row_count": int(len(jessi_rows))}


### 4. Create Jessi's master CSV and background audit files

The final cell reruns the live IPSA CSBS pipeline, validates the four-column master table, writes the supporting audit files, and displays only Jessi's simple assignment view.


In [4]:
def _atomic_write_csv(frame, path):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary_path, index=False)
    os.chmod(temporary_path, 0o600)
    os.replace(temporary_path, path)
    os.chmod(path, 0o600)


def _atomic_write_json(payload, path):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
    os.chmod(temporary_path, 0o600)
    os.replace(temporary_path, path)
    os.chmod(path, 0o600)


def write_outputs(master_df, assignments_df, trace_df, audit_df, workload_df, mapping_df, manifest):
    OUTPUT_DIR.mkdir(mode=0o700, parents=True, exist_ok=True)
    os.chmod(OUTPUT_DIR, 0o700)
    paths = {
        "Jessi master": OUTPUT_DIR / "IPSA_CSBS_scoring_assignments_master.csv",
        "Detailed assignments": OUTPUT_DIR / "csbs_scoring_assignments.csv",
        "Candidate trace": OUTPUT_DIR / "csbs_scoring_candidate_trace.csv",
        "Exclusions and audit": OUTPUT_DIR / "csbs_exclusions_audit.csv",
        "Workload summary": OUTPUT_DIR / "csbs_workload_summary.csv",
        "Field mapping": OUTPUT_DIR / "csbs_field_mapping_audit.csv",
        "Run manifest": OUTPUT_DIR / "csbs_run_manifest.json",
    }
    with TemporaryDirectory(prefix=".csbs-staging-", dir=OUTPUT_DIR) as staging_name:
        staging_dir = Path(staging_name)
        staged_paths = {key: staging_dir / path.name for key, path in paths.items()}
        _atomic_write_csv(assignments_df, staged_paths["Detailed assignments"])
        _atomic_write_csv(trace_df, staged_paths["Candidate trace"])
        _atomic_write_csv(audit_df, staged_paths["Exclusions and audit"])
        _atomic_write_csv(workload_df, staged_paths["Workload summary"])
        _atomic_write_csv(mapping_df, staged_paths["Field mapping"])
        _atomic_write_json(manifest, staged_paths["Run manifest"])
        _atomic_write_csv(master_df, staged_paths["Jessi master"])

        for label in (
            "Detailed assignments",
            "Candidate trace",
            "Exclusions and audit",
            "Workload summary",
            "Field mapping",
            "Run manifest",
        ):
            os.replace(staged_paths[label], paths[label])
        # Publish the Jessi-facing file last, after every staged artifact succeeds.
        os.replace(staged_paths["Jessi master"], paths["Jessi master"])

    for path in paths.values():
        os.chmod(path, 0o600)
    return paths


def run_ipsa_csbs_pipeline():
    token = os.getenv(TOKEN_ENV_VAR)
    if not token:
        raise EnvironmentError(
            f"{TOKEN_ENV_VAR} is not set. Supply the IPSA token through the environment."
        )

    run_started_at = datetime.now(timezone.utc)
    (
        metadata_df,
        instruments_df,
        form_event_mapping_df,
        repeating_df,
        repeating_status,
    ) = export_discovery_tables(token)
    csbs_events, scoring_events = validate_and_get_csbs_events(
        metadata_df, instruments_df, form_event_mapping_df
    )
    raw_df, examiner_field_returned, nonblank_examiner_rows = (
        export_minimal_csbs_records(token, csbs_events)
    )
    normalized = normalize_csbs_records(raw_df, csbs_events, scoring_events)
    reference_report_keys_df = export_reference_report_population(token)
    reference_report_validation = validate_reference_report_population(
        normalized["eligible"], reference_report_keys_df
    )
    mapping_df = build_field_mapping_audit(
        metadata_df,
        examiner_field_returned,
        nonblank_examiner_rows,
        normalized["counts"],
    )
    assignments_df, trace_df, workload_df = assign_scoring_clinicians(
        normalized["eligible"], normalized["contact_history"], seed=RANDOM_SEED
    )
    master_df = assignments_df[MASTER_COLUMNS].copy()
    replay_assignments_df, replay_trace_df, _ = assign_scoring_clinicians(
        normalized["eligible"], normalized["contact_history"], seed=RANDOM_SEED
    )

    counts = dict(normalized["counts"])
    counts["assigned_rows"] = int(len(assignments_df))
    counts["candidate_trace_rows"] = int(len(trace_df))
    checks = run_quality_checks(
        normalized,
        master_df,
        assignments_df,
        trace_df,
        workload_df,
        replay_assignments_df,
        replay_trace_df,
        examiner_field_returned,
        reference_report_validation,
    )
    jessi_result = (
        run_jessi_validation(assignments_df, trace_df)
        if JESSI_RECORD_ID
        else {"status": "not_run", "reason": "live record ID not supplied"}
    )
    blank_fallback_rows = int(counts["blank_examiner_fallback_rows"])
    run_status = (
        "completed_with_blank_examiner_fallback"
        if blank_fallback_rows
        else "completed"
    )
    limitations = {
        "examiner_field_returned": bool(examiner_field_returned),
        "nonblank_examiner_rows_in_csbs_export": int(nonblank_examiner_rows),
        "rows_where_no_self_assignment_is_verifiable": int(
            counts["known_examiner_eligible_rows"]
        ),
        "rows_where_no_self_assignment_is_not_verifiable": blank_fallback_rows,
        "full_blinding_guarantee": blank_fallback_rows == 0,
        "note": (
            "Blank-examiner assignments are workload-balanced and reproducible, "
            "but cannot prove that the assigned scorer differs from the unreported examiner."
            if blank_fallback_rows
            else "Examiner values were available for every assignment."
        ),
    }
    manifest = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "run_started_at_utc": run_started_at.isoformat(),
        "status": run_status,
        "primary_deliverable": "IPSA_CSBS_scoring_assignments_master.csv",
        "endpoint_hostname": urlparse(API_URL).hostname,
        "api_operations": [
            "project (connectivity only)",
            "metadata",
            "instrument",
            "formEventMapping",
            "repeatingFormsEvents (when available)",
            "record (read only)",
            f"report {REFERENCE_REPORT_ID} (read only; population verification)",
        ],
        "selected_forms": list(CSBS_FORM_NAMES),
        "selected_events": {
            "contact_history": csbs_events,
            "scoring_output": scoring_events,
        },
        "eligibility_reference": {
            "report_id": REFERENCE_REPORT_ID,
            "report_name": REFERENCE_REPORT_NAME,
            "report_scope": "base REDCap IDs after double-entry copies are ignored",
            "target_visit_months": list(TARGET_VISIT_MONTHS),
            "base_export_rows": int(
                reference_report_validation["reference_scope_export_rows"]
            ),
            "completed_target_double_entry_rows_ignored": int(
                counts["completed_target_dde_rows_ignored"]
            ),
            "live_report_row_count": int(reference_report_validation["row_count"]),
            "id_event_multiset_match": bool(
                reference_report_validation["id_event_multiset_match"]
            ),
        },
        "selected_fields": {
            "record_id": RECORD_ID_FIELD,
            "visit_month": "redcap_event_name",
            "examiner": EXAMINER_FIELD,
            "evaluation_date": EVALUATION_DATE_FIELD,
            "completion": COMPLETION_FIELD,
        },
        "double_entry_record_policy": {
            "source": RECORD_ID_FIELD,
            "dde_suffix_pattern": DDE_RECORD_ID_SUFFIX_PATTERN,
            "ignore": bool(IGNORE_DDE_RECORDS),
            "scope": (
                "IDs ending --1 or --2 are excluded from assignments and contact history"
            ),
        },
        "event_field_handling": EVENT_FIELD_HANDLING,
        "contact_history_policy": (
            "all completed CSBS events for base IDs only; one interaction per child, "
            "scoring clinician, and Date of Evaluation; double-entry IDs are ignored"
        ),
        "furthest_in_time_policy": (
            f"nearest absolute gap in days using {EVALUATION_DATE_FIELD}; dates are not imputed"
        ),
        "repeating_instance_handling": REPEATING_INSTANCE_HANDLING,
        "repeating_metadata_status": repeating_status,
        "repeating_metadata_rows": int(len(repeating_df)),
        "blank_examiner_policy": (
            "lowest current workload; random.Random(42) among tied lowest workloads"
        ),
        "examiner_export_verification": {
            "required_field": EXAMINER_FIELD,
            "field_returned": bool(examiner_field_returned),
            "raw_nonblank_rows": int(nonblank_examiner_rows),
            "policy": "fail closed when the column is omitted or all values are blank",
        },
        "examiner_interpretation": {
            "policy": (
                "preserve exact raw display value; recognize only the configured observed-value "
                "allowlist after Unicode NFKC, whitespace collapse, and casefold; exclude and "
                "credit every Emma/Tessa/Axie scorer named in composite values"
            ),
            "known_values_to_scoring_clinicians": {
                key: list(value) for key, value in KNOWN_EXAMINER_VALUE_TO_SCORERS.items()
            },
            "mapping_counts": {
                key: int(counts[key])
                for key in (
                    "recognized_examiner_rows",
                    "recognized_pool_examiner_rows",
                    "recognized_nonpool_examiner_rows",
                    "recognized_composite_examiner_rows",
                    "missing_examiner_rows",
                    "unexpected_examiner_rows",
                    "irrelevant_unexpected_examiner_rows",
                )
            },
        },
        "random_seed": RANDOM_SEED,
        "row_counts": counts,
        "assignment_rule_counts": {
            str(key): int(value)
            for key, value in assignments_df["Assignment Rule"].value_counts().items()
        },
        "workload": {
            str(row.Clinician): int(row.Assignments)
            for row in workload_df.itertuples(index=False)
        },
        "quality_checks": checks,
        "limitations": limitations,
        "jessi_validation": jessi_result,
        "version_metadata": {
            "pipeline": PIPELINE_VERSION,
            "python": sys.version.split()[0],
            "pandas": pd.__version__,
            "requests": requests.__version__,
        },
    }
    output_paths = write_outputs(
        master_df,
        assignments_df,
        trace_df,
        normalized["audit"],
        workload_df,
        mapping_df,
        manifest,
    )

    display(Markdown("#### Jessi's IPSA CSBS scoring-assignment table"))
    display(master_df.head(PREVIEW_ROWS))
    display(
        Markdown(
            f"**{len(master_df):,} completed CSBS visits saved to "
            f"`{output_paths['Jessi master']}`.**"
        )
    )
    if blank_fallback_rows:
        display(
            Markdown(
                "*Some individual examiner cells are genuinely blank in an otherwise "
                "verified examiner export; those rows use the workload-balanced fallback.*"
            )
        )
    return {
        "master_assignments": master_df,
        "assignments": assignments_df,
        "candidate_trace": trace_df,
        "exclusions_audit": normalized["audit"],
        "workload_summary": workload_df,
        "field_mapping_audit": mapping_df,
        "manifest": manifest,
        "quality_checks": checks,
        "output_paths": output_paths,
        "jessi_validation": jessi_result,
    }


ipsa_csbs_results = run_ipsa_csbs_pipeline()


,ID,Visit Month,Examiner,Assigned Scoring Clinician,Assignment Rule,Assigned Clinician Visit Count,Assigned Clinician Nearest Gap (days),Workload Before Assignment
0,1108,9,Emma Platt,Tessa,NS,0,<NA>,112
1,1108,12,Alexis Federico,Tessa,NS,0,<NA>,113
2,1108,15,Eilis McLaughlin,Tessa,NS,0,<NA>,114
3,1108,18,Axie Acosta,Tessa,NS,0,<NA>,115
4,1108,24,Eilis McLaughlin,Tessa,NS,0,<NA>,116


,ID,Visit Month,Examiner,Candidate,"Candidate Visit Count n(c,i)",Never Seen NS,Least Visits LV,Furthest in Time FT,"Nearest Gap d(c,r) (days)","Workload Before W(c,t)",Selected,Winning Rule
0,1108,9,Emma Platt,Tessa,0,True,False,False,<NA>,112,True,NS
1,1108,9,Emma Platt,Axie,1,False,False,False,270,115,False,NS
2,1108,12,Alexis Federico,Emma,1,False,False,False,97,112,False,NS
3,1108,12,Alexis Federico,Tessa,0,True,False,False,<NA>,113,True,NS
4,1108,12,Alexis Federico,Axie,1,False,False,False,173,115,False,NS
5,1108,15,Eilis McLaughlin,Emma,1,False,False,False,189,112,False,NS
6,1108,15,Eilis McLaughlin,Tessa,0,True,False,False,<NA>,114,True,NS
7,1108,15,Eilis McLaughlin,Axie,1,False,False,False,81,115,False,NS
8,1108,18,Axie Acosta,Emma,1,False,False,False,270,112,False,NS
9,1108,18,Axie Acosta,Tessa,0,True,False,False,<NA>,115,True,NS


Jessi rows and candidate trace shown. No pass/fail claim is made because no expected live assignment table was supplied.


#### Jessi's IPSA CSBS scoring-assignment table

,ID,Visit Month,Examiner,Assigned Scoring Clinician
0,1001,9,NaN,Tessa
1,1001,12,NaN,Axie
2,1001,15,NaN,Emma
3,1001,24,NaN,Axie
4,1002,9,NaN,Tessa
5,1002,12,NaN,Emma
6,1002,24,NaN,Axie
7,1003,9,NaN,Emma
8,1003,12,NaN,Tessa
9,1003,15,NaN,Axie


**486 completed CSBS visits saved to `csbs_redcap_outputs/IPSA_CSBS_scoring_assignments_master.csv`.**

*Some individual examiner cells are genuinely blank in an otherwise verified examiner export; those rows use the workload-balanced fallback.*

## Takeaways

Use the four-column master CSV only from a successful live run. It is published last, after double-entry exclusion checks, base-population REDCap report matching, examiner/date validation, contact deduplication, assignment, deterministic replay, and all quality checks pass.
